In [4]:
import sys
import os
current_path = os.path.abspath('')
data_path = '/'.join(current_path.split('/')[:-2]) + '/data/'
log_path = '/'.join(current_path.split('/')[:-1]) + '/logs_annotated/'
script_path = '/'.join(current_path.split('/')[:-1]) + '/scripts/'
sys.path.append(script_path)
sys.path.append('/home/fkunneman/.local/lib/python3.10/site-packages/')
sys.path.append('/usr/lib/python3/dist-packages/')

In [6]:
import torch
import gc
import importlib
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface.llms import HuggingFacePipeline
from langchain_community.vectorstores import Chroma
import chromadb
from chromadb.api.types import EmbeddingFunction, Documents, Embeddings

import instruction_agent
import generate_response

In [ ]:
### initialize huggingface
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()

In [ ]:
# ### Load judge llm
# tokenizer_judge = AutoTokenizer.from_pretrained('BramVanroy/fietje-2-instruct')
# tokenizer_judge.pad_token = tokenizer_judge.eos_token # Ensure padding token is set
# judgemodel = AutoModelForCausalLM.from_pretrained('BramVanroy/fietje-2-instruct', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")

# #tokenizer_judge = AutoTokenizer.from_pretrained('mistralai/Mistral-Small-3.1-24B-Instruct-2503')
# #tokenizer_judge.pad_token = tokenizer_judge.eos_token # Ensure padding token is set
# #judgemodel = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-Small-3.1-24B-Instruct-2503', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
# # set judge pipeline
# pipe = pipeline(
#     "text-generation",
#     model=judgemodel,
#     tokenizer=tokenizer_judge,
#     return_full_text=False,
#     #max_new_tokens=250, # Limit the number of generated tokens for concise responses
#     do_sample=False, # Enable sampling for more varied responses
#     temperature=0.3 # Control creativity
# )
# llm = HuggingFacePipeline(pipeline=pipe)

In [8]:
### Load models for chatbot and rag in memory
with open('/home/fkunneman/hf.txt') as fin:
    hf = fin.read().strip()
os.environ["HF_TOKEN"] = hf
torch.cuda.empty_cache()
gc.collect()
# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained('BramVanroy/GEITje-7B-ultra', use_fast=True)
tokenizer.pad_token = tokenizer.eos_token # Ensure padding token is set
# Initialize language model
chatmodel = AutoModelForCausalLM.from_pretrained('BramVanroy/GEITje-7B-ultra', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")
#chatmodel = AutoModelForCausalLM.from_pretrained('robinsmits/Qwen1.5-7B-Dutch-Chat', dtype=torch.bfloat16, trust_remote_code=True, device_map="cuda")     
# set pipeline
pipe = pipeline(
    "text-generation",
    model=chatmodel,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=250, # Limit the number of generated tokens for concise responses
    do_sample=True, # Enable sampling for more varied responses
    temperature=0.3 # Control creativity
)
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
chroma_client = chromadb.Client()
#embedding_model = SentenceTransformer('jegormeister/bert-base-dutch-cased')
collection = chroma_client.create_collection(
    name="instruction_db5",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 30
        }
    }
)

In [12]:
instructions_travel = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_ov_stripped.csv'
instructions_passport = data_path + 'opslag_inclusieve_spraakassistent_project/instructions_paspoort_stripped_v2.csv'
pat = data_path + 'opslag_inclusieve_spraakassistent_project/patterns_v2.csv'
qa_travel = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_ov_v5.csv'
qa_passport = data_path + 'opslag_inclusieve_spraakassistent_project/Vraag_antwoord_paspoort.csv'
nav = data_path + 'opslag_inclusieve_spraakassistent_project/navigation.csv'


In [22]:
importlib.reload(instruction_agent)
assistant = instruction_agent.InstructAgent(llm,'passport')
assistant.prepare_patterns(pat)
assistant.prepare_instructions(instructions_travel,'travel')
assistant.prepare_instructions(instructions_passport,'passport')
assistant.setup_rag(collection,[['qa','travel',qa_travel],['qa','passport',qa_passport],['nav',nav]])
assistant.initiate()

In [25]:
# initialize response generator
importlib.reload(sys.modules['generate_response'])
from generate_response import Turns, ResponseGenerator

turns = Turns()
turns.read_log(log_path + 'P1 conversatie paspoort annotated.csv')

rgen = ResponseGenerator(assistant)
for i,turn in enumerate(turns.turns):
    output = rgen.generate_response(turn,threshold=0.40)
    print('TURN',i,':',turn['asr'],',',output)

TURN 0 :  ik wil starta. , 
TURN 1 :  ik ben erold. , Je zegt ' ik ben erold.'. Weet je zeker dat je wil stoppen met het gesprek?
TURN 2 :  nee. , Dan wachten we nog even. Neem je tijd!
TURN 3 :  volgende , Stap 4. Onder 'persoon/personen' kun je kiezen voor hoeveel personen je het paspoort aanvraagt. Druk op de 'plus-knop' om mensen toe te voegen aan het witte vakje.


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TURN 4 :  heb ik al gedaan. , Er is deze maand geen afspraak meer mogelijk. 


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 10.033668882000256
TURN 5 :  zoeken naar paspoort. ,  Je kunt de sectie 'Aanvragen' vinden onder de blauwe tekst 'Paspoort 18 jaar en ouder'.





Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 7.343098011000166
TURN 6 :  ik zie het niet. ,  Geen probleem, blijf rustig. De sectie 'Aanvragen' staat meestal onder de tekst waar je op hebt geklikt, in dit geval 'Paspoort 18 jaar en ouder'.



Prompt took 10.116253337999751
TURN 7 :  *** ,  Als je een asterisk (*) ziet, betekent dit dat er een probleem is met de website of dat je een verkeerde stap hebt genomen. Probeer terug te gaan naar de vorige stap en de instructies nog eens te volgen
TURN 8 :  ik zie niet waar ik het moet aanvragen. , Het veld 'voornaam' staat onder het veld 'achternaam'


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 10.182636491999801
TURN 9 :  exie het niet. ,  Je kunt het veld 'voornaam' vinden onder het veld 'achternaam'. Het staat meestal direct onder het veld 'achternaam'.



Prompt took 10.229759888999979
TURN 10 :  salik doon. ,  Goed, je kunt de veld 'voornaam' invullen nadat je de 'achternaam' hebt ingevuld.



TURN 11 :  volgende stap. , Klik op de knop '2 Kies een locatie' en Kies de locatie voor je afspraak


Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt took 10.278658570000061
TURN 12 :  paspoortaanvragen ,  Als je vragen hebt over de paspoortaanvragen, kun je de website van de gemeente Amsterdam bezoeken of contact opnemen met de klantenservice van de gemeente voor meer informatie.




Prompt took 10.299370455999906
TURN 13 :  waar staat de datum? ,  De datum waarop je de afspraak wilt maken, kun je invoeren in het veld 'Datum afspraak'. Dit veld bevindt zich meestal naast het veld 'Beschikbare tijden'.



TURN 14 :  ik kom er niet bij. , De knop '2 Kies een locatie' staat onder de plus knop
